In [1]:
import os
import gc
import numpy as np
import tensorflow as tf
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# 1. Plotting Configuration
# ==============================================================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 300 
})

# ==============================================================================
# 2. Custom Layers Registration (Crucial for loading Transformers)
# ==============================================================================
@tf.keras.utils.register_keras_serializable(name="Patches")
class Patches(tf.keras.layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        return tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

def get_sinusoidal_position_embedding(sequence_length, dim):
    position = np.arange(sequence_length)[:, np.newaxis]
    div_term = np.exp(np.arange(0, dim, 2) * -(np.log(10000.0) / dim))
    pe = np.zeros((sequence_length, dim))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return tf.constant(pe, dtype=tf.float32)

@tf.keras.utils.register_keras_serializable(name="SinusoidalPositionEmbedding")
class SinusoidalPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, dim, **kwargs):
        super(SinusoidalPositionEmbedding, self).__init__(**kwargs)
        self.sequence_length = sequence_length
        self.dim = dim
        self.pos_embedding = get_sinusoidal_position_embedding(sequence_length, dim)

    def call(self, x):
        return x + self.pos_embedding

    def get_config(self):
        config = super(SinusoidalPositionEmbedding, self).get_config()
        config.update({"sequence_length": self.sequence_length, "dim": self.dim})
        return config

@tf.keras.utils.register_keras_serializable(name="PatchEncoder")
class PatchEncoder(tf.keras.layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = tf.keras.layers.Dense(units=projection_dim)
        pos_encoding_matrix = get_sinusoidal_position_embedding(num_patches, projection_dim)
        self.position_embedding = tf.Variable(pos_encoding_matrix, trainable=False, name="position_embedding")

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        return encoded + self.position_embedding 

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# ==============================================================================
# 3. Configuration & Paths
# ==============================================================================
MODELS_DIR = "models" 
DATA_BASE_DIR = "."

CHANNEL_NAMES = [
    'F3', 'Fz', 'F4', 'FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 
    'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 
    'CPz', 'CP2', 'CP4', 'CP6', 'P3', 'Pz', 'P4'
]

BRAIN_REGIONS = {
    'Frontal (EOG/Noise Risk)': ['F3', 'Fz', 'F4'],
    'Pre-Motor': ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6'],
    'Sensorimotor (Target)': ['C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6'],
    'Parietal': ['P3', 'Pz', 'P4']
}

DATASETS = ["Raw", "Filtered_8_30Hz", "Filtered_8_13Hz"]
MODELS = ["eegnet", "cvt", "vit_pe"]

# ==============================================================================
# 4. Core Saliency Functions
# ==============================================================================
@tf.function
def compute_gradients(model, input_tensor):
    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        preds = model(input_tensor, training=False)
        # Compute gradients w.r.t. the predicted class (highest logit/probability)
        top_class = tf.argmax(preds[0])
        loss = preds[:, top_class]
    return tape.gradient(loss, input_tensor)

def extract_importance(model, data_folder, model_type, num_samples=150):
    all_files = [f for f in os.listdir(data_folder) if f.endswith('.npy')]
    np.random.shuffle(all_files)
    files_to_process = all_files[:min(num_samples, len(all_files))]
    
    cumulative_saliency = np.zeros(27)
    valid_count = 0
    
    for file in tqdm(files_to_process, desc="Extracting Saliency", leave=False):
        try:
            sample = np.load(os.path.join(data_folder, file))
            
            # Input shape standardization across architectures
            if model_type in ["cvt", "vit_pe"]:
                # Transformer formats: (Timepoints, Channels) -> (2560, 27)
                if sample.shape == (27, 2560): sample = sample.T
                input_data = sample[np.newaxis, ...]
            else:
                # EEGNet format: (Channels, Timepoints, 1) -> (27, 2560, 1)
                if sample.shape == (2560, 27): sample = sample.T
                input_data = sample[np.newaxis, ..., np.newaxis]
                
            input_tensor = tf.convert_to_tensor(input_data, dtype=tf.float32)
            grads = compute_gradients(model, input_tensor)
            saliency = np.abs(grads.numpy()[0])
            
            # Average saliency across temporal dimension to derive spatial channel importance
            if model_type in ["cvt", "vit_pe"]:
                # Saliency shape: (2560, 27) -> average along time dimension (axis 0)
                channel_importance = np.mean(saliency, axis=0)
            else:
                # Saliency shape: (27, 2560, 1) -> average along time dimension (axis 1)
                channel_importance = np.mean(saliency, axis=1).flatten()
                
            cumulative_saliency += channel_importance
            valid_count += 1
        except Exception as e:
            continue
            
    if valid_count == 0:
        return np.zeros(27)
    return cumulative_saliency / valid_count

# ==============================================================================
# 5. Main Execution
# ==============================================================================
print("🚀 Starting Comprehensive Saliency Analysis...\n")
summary_list = []
os.makedirs("XAI_Figures", exist_ok=True)

for dataset in DATASETS:
    # Dataset path (assuming .npy files reside directly within dataset subdirectories)
    data_folder = os.path.join(DATA_BASE_DIR, dataset)
    if not os.path.exists(data_folder):
        print(f"[SKIP] Data folder not found: {data_folder}")
        continue

    for model_prefix in MODELS:
        # Evaluate representative Fold 1 checkpoints for computational efficiency
        model_file = f"{model_prefix}_{dataset}_fold1.h5"
        model_path = os.path.join(MODELS_DIR, model_file)
        
        if not os.path.exists(model_path): 
            print(f"[SKIP] Model not found: {model_file}")
            continue
            
        try:
            model = tf.keras.models.load_model(
                model_path, 
                custom_objects={
                    'Patches': Patches, 
                    'SinusoidalPositionEmbedding': SinusoidalPositionEmbedding,
                    'PatchEncoder': PatchEncoder
                }, 
                compile=False
            )
            
            print(f"\n{'-'*60}")
            print(f"🔍 ANALYZING: {model_prefix.upper()} on {dataset}")
            print(f"{'-'*60}")
            
            importance = extract_importance(model, data_folder, model_prefix, num_samples=150)
            
            total_importance = np.sum(importance)
            if total_importance == 0: continue
            percentages = (importance / total_importance) * 100
                
            # Display top-ranked electrodes
            sorted_indices = np.argsort(percentages)[::-1]
            print("\nTop 5 Electrodes:")
            top_names = []
            top_vals = []
            for i in range(10):
                idx = sorted_indices[i]
                top_names.append(CHANNEL_NAMES[idx])
                top_vals.append(percentages[idx])
                if i < 5: print(f"  {CHANNEL_NAMES[idx]:<4} : {percentages[idx]:.2f}%")
                
            # Aggregate weights across neuroanatomical regions
            region_dict = {"Model": model_prefix.upper(), "Dataset": dataset}
            region_names = []
            region_vals = []
            
            for region_name, channels in BRAIN_REGIONS.items():
                region_sum = sum(percentages[CHANNEL_NAMES.index(c)] for c in channels)
                short_name = region_name.split(' (')[0]
                region_names.append(short_name)
                region_vals.append(region_sum)
                region_dict[region_name] = round(region_sum, 2)
                
            summary_list.append(region_dict)
            
            # --- Visualization & Figure Export ---
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle(f'Spatial Attention: {model_prefix.upper()} on {dataset}', fontsize=16, fontweight='bold', y=1.05)
            
            sns.barplot(x=top_vals, y=top_names, ax=axes[0], palette="viridis", hue=top_names, legend=False)
            axes[0].set_title('Top 10 Electrodes', pad=10)
            axes[0].set_xlabel('Importance Weight (%)')
            
            sns.barplot(x=region_names, y=region_vals, ax=axes[1], palette="magma", hue=region_names, legend=False)
            axes[1].set_title('Regional Distribution', pad=10)
            axes[1].set_ylabel('Cumulative Weight (%)')
            
            plt.tight_layout()
            plot_path = os.path.join("XAI_Figures", f"Saliency_{model_prefix}_{dataset}.png")
            plt.savefig(plot_path, bbox_inches='tight')
            plt.close()
            
            del model
            tf.keras.backend.clear_session()
            gc.collect()
            
        except Exception as e:
            print(f"\n[ERROR] Failed processing {model_file}. Details: {str(e)}")

# ==============================================================================
# 6. Save Summary Report
# ==============================================================================
if summary_list:
    df = pd.DataFrame(summary_list)
    csv_path = os.path.join("XAI_Figures", "Brain_Region_Weights_Summary.csv")
    df.to_csv(csv_path, index=False)
    print(f"\n{'-'*60}")
    print(f"✅ Analysis complete! Summary saved to: {csv_path}")
    print(df.to_string(index=False))

🚀 Starting Comprehensive Saliency Analysis...


------------------------------------------------------------
🔍 ANALYZING: EEGNET on Raw
------------------------------------------------------------



Top 5 Electrodes:
  P4   : 15.61%
  CP3  : 11.06%
  P3   : 7.31%
  FC6  : 5.81%
  FCz  : 5.17%

------------------------------------------------------------
🔍 ANALYZING: CVT on Raw
------------------------------------------------------------



Top 5 Electrodes:
  FCz  : 5.94%
  FC6  : 5.89%
  P4   : 5.61%
  P3   : 4.79%
  C2   : 4.71%

------------------------------------------------------------
🔍 ANALYZING: VIT_PE on Raw
------------------------------------------------------------



Top 5 Electrodes:
  P4   : 6.68%
  CP3  : 6.53%
  C2   : 5.58%
  CP4  : 5.16%
  C5   : 4.66%

------------------------------------------------------------
🔍 ANALYZING: EEGNET on Filtered_8_30Hz
------------------------------------------------------------



Top 5 Electrodes:
  FC6  : 14.55%
  P4   : 8.85%
  F3   : 5.81%
  C6   : 5.72%
  CP3  : 5.04%

------------------------------------------------------------
🔍 ANALYZING: CVT on Filtered_8_30Hz
------------------------------------------------------------



Top 5 Electrodes:
  P4   : 10.25%
  CP3  : 9.68%
  FCz  : 6.61%
  FC3  : 5.67%
  FC6  : 4.91%

------------------------------------------------------------
🔍 ANALYZING: VIT_PE on Filtered_8_30Hz
------------------------------------------------------------



Top 5 Electrodes:
  P4   : 8.38%
  CP3  : 6.69%
  FC3  : 4.34%
  FC6  : 3.90%
  FCz  : 3.84%

------------------------------------------------------------
🔍 ANALYZING: EEGNET on Filtered_8_13Hz
------------------------------------------------------------



Top 5 Electrodes:
  CP3  : 7.01%
  C2   : 5.91%
  CP2  : 5.77%
  P4   : 5.65%
  CP1  : 5.54%

------------------------------------------------------------
🔍 ANALYZING: CVT on Filtered_8_13Hz
------------------------------------------------------------



Top 5 Electrodes:
  P4   : 9.10%
  CP3  : 8.29%
  FC3  : 7.67%
  FCz  : 6.17%
  FC6  : 5.51%

------------------------------------------------------------
🔍 ANALYZING: VIT_PE on Filtered_8_13Hz
------------------------------------------------------------



Top 5 Electrodes:
  P4   : 8.27%
  CP3  : 6.33%
  FC3  : 5.13%
  FCz  : 5.10%
  FC6  : 4.81%

------------------------------------------------------------
✅ Analysis complete! Summary saved to: XAI_Figures\Brain_Region_Weights_Summary.csv
 Model         Dataset  Frontal (EOG/Noise Risk)  Pre-Motor  Sensorimotor (Target)  Parietal
EEGNET             Raw                      7.04      27.20                  41.53     24.24
   CVT             Raw                     10.56      28.75                  47.34     13.35
VIT_PE             Raw                      7.50      21.53                  56.86     14.11
EEGNET Filtered_8_30Hz                     11.18      34.30                  39.17     15.36
   CVT Filtered_8_30Hz                     10.84      31.08                  41.27     16.81
VIT_PE Filtered_8_30Hz                     10.22      25.80                  48.26     15.72
EEGNET Filtered_8_13Hz                      8.76      22.57                  59.02      9.65
   CVT Filtered_